<a href="https://colab.research.google.com/github/SakshamPainuly/underwater-creature-tracking/blob/anu%2Fmodel-development/notebooks/colab_train_rtdetr.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# UWCT — Colab RT-DETR training (E003, YOLO vs RT-DETR comparison)

Same pipeline as `notebooks/colab_train_baseline.ipynb` (YOLO E001), run for **E003**
(`rtdetr-l.pt`, D-015). Runbook: `docs/COLAB.md`. This notebook only calls repository scripts.

- Runtime: **GPU** (Runtime → Change runtime type). Never run training on the local PC.
- Same frozen dataset **v1** view as E001 (YOLO-format labels, no conversion needed);
  zExtra never reaches training; the training data file has **no test split**.
- `train_yolo.py` detects the RT-DETR checkpoint and uses Ultralytics' `RTDETR` class.
- Runs, checkpoints and records go to `MyDrive/uwct/runs/E003*/` (outside Git).
- Before the full run: the smoke test (cell 12) must finish **without NaN losses** and within GPU
  memory at batch 16. If not, stop — changing batch or AMP needs a new decision (D-015).
- The **FINAL TEST EVALUATION** section runs only after E003 is registered, reviewed and declared
  in `experiments/final_candidates.csv`.

In [ ]:
# 1. Settings -- set COMMIT (full 40-character sha of the commit to run); keep everything else.
import os, re, subprocess, sys

REPO = "SakshamPainuly/underwater-creature-tracking"
BRANCH = "anu/model-development"
COMMIT = "8989077f595c3d14c04bb35d1cba7410b3fa3f0c"  # REQUIRED: `git rev-parse HEAD` on the local PC, after that commit is pushed
EXPERIMENT_CONFIG = "configs/experiments/E003_rtdetr_baseline.yaml"
EXPERIMENT_ID = "E003"
TOKEN_SECRET_NAME = "UWCT_GITHUB_TOKEN"  # Colab Secrets: fine-grained, this repo only, Contents read-only

REPO_DIR = "/content/uwct"                             # fixed: resumed runs expect the same dataset path
DRIVE_RAW = "/content/drive/MyDrive/uwct/raw/biology"  # Set 100 + Set 2000 only (no zExtra)
RUNS_ROOT = "/content/drive/MyDrive/uwct/runs"         # persistent run folders
LOCAL_RAW = "/content/raw/biology"                     # fast local copy used for training
WEIGHTS_DIR = "/content/weights"                       # pretrained checkpoints downloaded by Ultralytics

assert re.fullmatch(r"[0-9a-f]{40}", COMMIT), "Set COMMIT to the exact 40-character commit sha to run"


def sh(*cmd):
    """Run a command in the repo, stream its output, raise if it fails."""
    p = subprocess.Popen(list(cmd), cwd=REPO_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="")
    if p.wait() != 0:
        raise RuntimeError(f"FAILED (exit {p.returncode}): {' '.join(cmd)}")

In [ ]:
# 2. GPU present?
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout or "NO GPU -- switch the runtime to GPU")

In [ ]:
# 3. Mount Google Drive (raw training subset + persistent runs folder)
from google.colab import drive
drive.mount("/content/drive")
os.makedirs(RUNS_ROOT, exist_ok=True)

In [ ]:
# 4. Check out the exact commit (read-only token from Colab Secrets; see docs/COLAB.md).
#    Git asks the GIT_ASKPASS helper for credentials; the helper answers from the environment of
#    that one git process. The token is read from Colab Secrets inside git_remote() at call time
#    and is never in a command-line argument, URL, file, .git/config or output.
import stat
from google.colab import userdata

REPO_URL = f"https://github.com/{REPO}.git"  # no credentials in the URL
ASKPASS = "/content/uwct_git_askpass.sh"      # holds no secret; printf/echo are shell builtins
with open(ASKPASS, "w") as f:
    f.write('#!/bin/sh\n'
            'case "$1" in\n'
            '  Username*) echo "x-access-token" ;;\n'
            '  *) printf "%s\\n" "$UWCT_GIT_TOKEN" ;;\n'
            'esac\n')
os.chmod(ASKPASS, stat.S_IRWXU)


def git(*args, cwd=REPO_DIR):
    """Local git commands: no credentials involved."""
    r = subprocess.run(["git", *args], cwd=cwd, capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(f"git {args[0]} failed:\n{r.stderr}")
    return r.stdout.strip()


def git_remote(*args, cwd=REPO_DIR):
    """clone/fetch: the token exists only in this git child's environment, for this call only."""
    env = {**os.environ, "GIT_ASKPASS": ASKPASS, "GIT_TERMINAL_PROMPT": "0",
           "UWCT_GIT_TOKEN": userdata.get(TOKEN_SECRET_NAME)}
    try:
        # credential.helper= (empty) disables every credential helper, so nothing is stored
        r = subprocess.run(["git", "-c", "credential.helper=", *args], cwd=cwd, env=env,
                           capture_output=True, text=True)
        err = r.stderr.replace(env["UWCT_GIT_TOKEN"], "***")
        config = os.path.join(REPO_DIR, ".git", "config")
        leaked = os.path.isfile(config) and env["UWCT_GIT_TOKEN"] in open(config).read()
    finally:
        env.clear()
    if r.returncode != 0:
        raise RuntimeError(f"git {args[0]} failed:\n{err}")
    assert not leaked, "token found in .git/config"


if not os.path.isdir(REPO_DIR):
    git_remote("clone", "--branch", BRANCH, REPO_URL, REPO_DIR, cwd="/content")
else:  # reconnect / new commit: fetch the branch again
    git_remote("fetch", "origin", BRANCH)
git("checkout", "--detach", COMMIT)
assert git("rev-parse", "HEAD") == COMMIT
assert git("remote", "get-url", "origin") == REPO_URL
print("commit:", COMMIT, "| working tree:", git("status", "--porcelain") or "clean")

In [ ]:
# 5. Install the pinned GPU requirements (ultralytics==8.4.155; torch/CUDA come from Colab)
sh(sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-gpu.txt")
sh(sys.executable, "-c", "import ultralytics, torch; print('ultralytics', ultralytics.__version__, "
   "'| torch', torch.__version__, '| cuda', torch.version.cuda)")

In [ ]:
# 6. Environment report for this session (each run record stores its own copy too)
sh(sys.executable, "scripts/env_report.py", "--out", "/content/env_report_session.json")

In [ ]:
# 7. Reconstruct Biology raw data locally from the shared ZIP.
#    Nothing is written back to Google Drive.

import os
import shutil
import zipfile
import subprocess
import sys
from pathlib import Path

ZIP_ID = "1_97E8IGX_euJLAzXlyfpukhvRjg2XhfM"
ZIP_PATH = Path("/content/biology_dataset.zip")
LOCAL_RAW = Path("/content/raw/biology")

# Start clean
if LOCAL_RAW.exists():
    shutil.rmtree(LOCAL_RAW)

LOCAL_RAW.mkdir(parents=True, exist_ok=True)

# Install gdown
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "gdown"],
    check=True
)

# Download shared ZIP
print("Downloading Biology ZIP...")
subprocess.run(
    [
        sys.executable,
        "-m",
        "gdown",
        f"https://drive.google.com/uc?id={ZIP_ID}",
        "-O",
        str(ZIP_PATH),
    ],
    check=True
)

print("Downloaded:", ZIP_PATH)
print("Size:", ZIP_PATH.stat().st_size, "bytes")

# Extract outer ZIP
OUTER = Path("/content/biology_outer")
if OUTER.exists():
    shutil.rmtree(OUTER)
OUTER.mkdir()

print("\nExtracting outer ZIP...")
with zipfile.ZipFile(ZIP_PATH, "r") as z:
    z.extractall(OUTER)

ROOT = OUTER / "AI Lobster Training Dataset"

# Set 100
print("\nCopying Set 100...")
shutil.copytree(
    ROOT / "Set 100",
    LOCAL_RAW / "Set 100"
)

# Set 2000 images
IMG_ZIP = ROOT / "Set 2000" / "batch_2 (2k - imgOnly).zip"
IMG_DST = LOCAL_RAW / "Set 2000" / "batch_2 (2k - imgOnly)"

print("Extracting Set 2000 images...")
IMG_DST.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(IMG_ZIP, "r") as z:
    z.extractall(IMG_DST)

# Set 2000 labels
LABEL_ZIP = ROOT / "Set 2000" / "batch_2_2k_labels.zip"
LABEL_DST = LOCAL_RAW / "Set 2000" / "batch_2_2k_labels"

print("Extracting Set 2000 labels...")
LABEL_DST.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(LABEL_ZIP, "r") as z:
    z.extractall(LABEL_DST)

# zExtra stays in raw data for fingerprint verification,
# but remains excluded from the frozen training view.
print("Copying zExtra Photos...")
shutil.copytree(
    ROOT / "zExtra Photos",
    LOCAL_RAW / "zExtra Photos"
)

# Project documents
for name in [
    "Lobster AI Detection Training Data.docx",
    "What's been done.xlsx",
]:
    src = ROOT / name
    if src.exists():
        shutil.copy2(src, LOCAL_RAW / name)

# Tell all subsequent project scripts where raw data lives
os.environ["UWCT_RAW_ROOT"] = str(LOCAL_RAW)

print("\n=== RAW ROOT ===")
print(LOCAL_RAW)

print("\n=== TOP LEVEL ===")
for p in sorted(LOCAL_RAW.iterdir()):
    print(p.name)

assert (LOCAL_RAW / "Set 100").is_dir()
assert (LOCAL_RAW / "Set 2000").is_dir()
assert (LOCAL_RAW / "zExtra Photos").is_dir()

print("\nCell 7 reconstruction complete.")

In [ ]:
# 8. Raw-data check of the local copy (structural: paths + sizes; txt contents hashed). Must MATCH.
sh(sys.executable, "scripts/check_raw_unchanged.py", "--root", LOCAL_RAW, "--subset", "training")

In [ ]:
# 9. Build the YOLO view of frozen v1 (verifies FROZEN.json hashes first; symlinks to the local copy)
sh(sys.executable, "scripts/prepare_yolo_dataset.py", "--version", "v1")

In [ ]:
# 10. Ultralytics reads the view exactly as built. Must print VERIFIED.
sh(sys.executable, "scripts/verify_yolo_dataset.py", "--dataset", "data/yolo/lobster_v1")

In [ ]:
# commit check
import subprocess

print("Current commit:")
print(
    subprocess.check_output(
        ["git", "rev-parse", "HEAD"],
        cwd="/content/uwct",
        text=True
    ).strip()
)

print("\nCandidate declaration:")
print(
    open(
        "/content/uwct/experiments/final_candidates.csv",
        encoding="utf-8"
    ).read()
)

In [ ]:
# 11. Preflight: every training check + load (download) the exact pretrained checkpoint.
#     If rtdetr-l.pt is unavailable in ultralytics 8.4.155 this STOPS -- report it; do not change the model.
TRAIN = [sys.executable, "scripts/train_yolo.py", "--config", EXPERIMENT_CONFIG,
         "--runs-root", RUNS_ROOT, "--weights-dir", WEIGHTS_DIR]
sh(*TRAIN, "--preflight-only")

In [ ]:
# 12. Smoke test: 1 epoch on 10% of train, validated on val -> RUNS_ROOT/E003_smoke_<utc>/
RUN_SMOKE = False  # set to True deliberately
assert RUN_SMOKE, "Smoke test not enabled"
sh(*TRAIN, "--smoke")

In [ ]:
# 12b. Smoke-test check (RT-DETR): losses must be finite before any full run (Ultralytics notes
#      AMP can give NaN for RT-DETR). Reads the newest E003 smoke run's results.csv on Drive.
import csv, glob, math

smoke_dirs = sorted(glob.glob(os.path.join(RUNS_ROOT, EXPERIMENT_ID + "_smoke_*")))
assert smoke_dirs, "no smoke run found -- run cell 12 first"
with open(os.path.join(smoke_dirs[-1], "train", "results.csv"), newline="", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))
last = {k.strip(): v for k, v in rows[-1].items()}
print(smoke_dirs[-1])
for k, v in last.items():
    print(f"{k:>28}: {v}")
bad = [k for k, v in last.items() if "loss" in k and not math.isfinite(float(v))]
assert not bad, f"non-finite losses {bad}: STOP -- record a decision before changing batch/amp (D-015)"
print("Smoke losses are finite.")

In [ ]:
# 13. Full training run -> RUNS_ROOT/E003/. Only after the smoke test passed and the run is approved.
RUN_FULL = False  # set to True deliberately
assert RUN_FULL, "Full training not enabled"
sh(*TRAIN)

In [ ]:
# 14. Resume after a Colab disconnect: re-run cells 1-10 (same COMMIT), then this cell.
RESUME = False  # set to True deliberately
assert RESUME, "Resume not enabled"
sh(*TRAIN, "--resume")

In [ ]:
# 15. Summary of the run record. Then download run_record.json and, on the local PC:
#     python scripts/register_run.py <path>/run_record.json
import json

rec = json.load(open(os.path.join(RUNS_ROOT, EXPERIMENT_ID, "run_record.json"), encoding="utf-8"))
print(json.dumps({k: rec[k] for k in ("run_name", "git", "seed", "ultralytics_version", "val_metrics")}, indent=2))
print("dataset:", rec["dataset"]["version"], rec["dataset"]["fingerprint"])
print("best:", rec["outputs"]["best"])

# FINAL TEST EVALUATION

The **only** place the test split is used (CLAUDE.md rule 7). Run it once per pre-declared final
candidate, after its training run is finished, registered and reviewed. The results never feed back
into training, thresholds or model choice.

Before running:
1. Add the experiment to `experiments/final_candidates.csv`, and declare it in `docs/EXPERIMENTS.md`.
2. Commit, push, and set `COMMIT` (cell 1) to that commit.
3. Run cells 1–10 in this session.

Afterwards, log the evaluation in `docs/EXPERIMENTS.md` (Test-evaluation log).

In [ ]:
# 16. FINAL TEST EVALUATION -- the only cell that touches the test split.
#     scripts/final_test_eval.py -> src/training/final_test.py:
#       a. operating confidence chosen on VAL (train/val data file), frozen;
#       b. RTDETR(best.pt).val(data=<full data.yaml incl. test>, split="test", plots=True);
#       c. prediction examples (one test image per test session) at the val-chosen confidence.
#     Everything is written to RUNS_ROOT/<experiment>/final_test/ on Drive (outside Git).
#     Refused unless the experiment is declared in experiments/final_candidates.csv at COMMIT.
FINAL_TEST_EXPERIMENT = EXPERIMENT_ID
RUN_FINAL_TEST = False  # set to True deliberately, once, after the declaration is committed
assert RUN_FINAL_TEST, "Final test evaluation not enabled"
print("=" * 72 + "\nFINAL TEST EVALUATION -- " + FINAL_TEST_EXPERIMENT + "\n" + "=" * 72)
sh(sys.executable, "scripts/final_test_eval.py", "--experiment", FINAL_TEST_EXPERIMENT,
   "--runs-root", RUNS_ROOT, "--confirm-final-test")

In [ ]:
# 17. FINAL TEST EVALUATION -- results and plots, read back from the run folder on Drive.
import glob
import json
from IPython.display import Image, Markdown, display

FINAL_DIR = os.path.join(RUNS_ROOT, FINAL_TEST_EXPERIMENT, "final_test")
rec = json.load(open(os.path.join(FINAL_DIR, "final_test_record.json"), encoding="utf-8"))
rep, op = rec["metrics"]["ultralytics_reported"], rec["metrics"]["at_val_conf"]
display(Markdown(f"# FINAL TEST EVALUATION — {rec['experiment_id']} (split = `{rec['split']}`, "
                 f"attempt {rec['attempts_including_this_one']})"))
rows = ["| metric | value |", "|---|---|",
        f"| mAP50 | {rep['map50']:.4f} |", f"| mAP50-95 | {rep['map50_95']:.4f} |"]
if op:
    rows += [f"| precision @ val-chosen conf {op['conf']:.3f} | {op['precision']:.4f} |",
             f"| recall @ val-chosen conf {op['conf']:.3f} | {op['recall']:.4f} |",
             f"| F1 @ val-chosen conf {op['conf']:.3f} | {op['f1']:.4f} |"]
rows += [f"| precision (Ultralytics: max-F1 conf on test) | {rep['precision']:.4f} |",
         f"| recall (Ultralytics: max-F1 conf on test) | {rep['recall']:.4f} |"]
display(Markdown("\n".join(rows)))
print("confusion matrix (rows = predicted, cols = true; lobster, background):",
      (rec["metrics"]["confusion_matrix"] or {}).get("matrix"))

PLOTS = [("Confusion matrix", ["confusion_matrix.png", "confusion_matrix_normalized.png"]),
         ("Precision-recall curve", ["BoxPR_curve.png", "PR_curve.png"]),
         ("F1 curve", ["BoxF1_curve.png", "F1_curve.png"])]
for title, names in PLOTS:
    found = [os.path.join(FINAL_DIR, "test", n) for n in names if os.path.isfile(os.path.join(FINAL_DIR, "test", n))]
    display(Markdown(f"### {title}" + ("" if found else " — not produced by this Ultralytics version")))
    for p in found:
        display(Image(filename=p, width=700))

display(Markdown(f"### Prediction examples at the val-chosen confidence ({op['conf']:.3f})" if op else
                 "### Prediction examples"))
for p in sorted(glob.glob(os.path.join(FINAL_DIR, "predictions", "*.*"))):
    display(Image(filename=p, width=900))
display(Markdown("### Ultralytics validation batches (labels vs predictions with confidence)"))
for p in sorted(glob.glob(os.path.join(FINAL_DIR, "test", "val_batch*_pred.jpg"))):
    display(Image(filename=p, width=900))
display(Markdown("**Limitations:**\n" + "\n".join(f"- {l}" for l in rec["limitations"])))